In [1]:
import MDAnalysis as md
import pandas as pd
import numpy as np
from scipy.spatial import KDTree
import itertools
from tqdm import tqdm

from scipy.interpolate import RegularGridInterpolator

#This one is for timing the approach
from datetime import datetime

In [5]:
'''
    Setup the md analysis universe
'''

system_psf = "../data/test/system_complete.psf" #For now, we consider the full system
system_pdb = "../data/test/system_complete.pdb"

system_universe = md.Universe(system_psf, system_pdb)
system = system_universe.select_atoms("not water and not name CLA POT")

In [6]:
'''
    Grid related variables
'''

bulk_conductivity = 10.5 #S/m 1M KCl experimental (assuming 1 atm & ~295K)

#These are from the PBC simulation I use (all in A)
size_x = 59
size_y = size_x #x & y should have same size
size_z = 85

#Atom Radii (in A)
radii_dict = dict(zip(pd.read_csv("../data/radial.csv")["Atom_type"], pd.read_csv("../data/radial.csv")["radius(A)"]))

#Radial linear conductance variables (from https://pubs.acs.org/doi/10.1021/acssensors.8b01375)
r_min = 1.4 #A
r_slope_inv = 2.9 #A (this should be divided for actual slope)

#Useful intermediate vars
r_intercept = -1.0 * r_min / r_slope_inv
r_max = r_slope_inv + r_min

#For conductivity map, just use np.clip(dist / r_slope_inv + r_intercept, 0, 1)

In [7]:
#Write the radii for each atom and add as attribute to 
names = np.strings.slice(system.names.astype(np.dtypes.StringDType), 0,1)
radii = np.array([radii_dict[k] for k in names])

In [149]:
'''
    Construct the grid and add the conductances based on the closest atom
'''

#Grid spacing along dimensions
x_arr = np.arange(-1.0 * int(size_x / 2), 1.0 * int(size_x / 2), 1, dtype=int)
y_arr = np.arange(-1.0 * int(size_y / 2), 1.0 * int(size_y / 2), 1, dtype=int)
z_arr = np.arange(-1.0 * int(size_z / 2), 1.0 * int(size_z / 2), 1, dtype=int)

#Will contain the conductances
grid = np.empty((len(x_arr), len(y_arr), len(z_arr)))

#Method for having nice tqdm progress bar
coords = list(itertools.product(x_arr,y_arr,z_arr))
pg = tqdm(coords)

#Generate the KDTree
kdt = KDTree(system.positions)

#Iterate through grid points
for x,y,z in pg:

    #Setup position
    pos = (x,y,z)

    #Do KDTree query
    query = kdt.query_ball_point(pos, r=(r_max + 1.9)) #1.9 comes from the radial.csv (will be the maximum distance in which conductance isn't just bulk)
    query_pos = system.positions[query]
    
    #If there are no neighbors within the maximum distance range, then set to bulk
    if(len(query) == 0):
        grid[x,y,z] = bulk_conductivity
    else:
        #Compute the distance for all possible points and readjust using the atom's radius
        query_dist = np.linalg.norm(query_pos - pos, axis=1)
        real_dist = query_dist - radii[query]
        selected_dist = np.min(real_dist)

        #Set the conductivity according to the linear piecewise as in the paper
        grid[x,y,z] = bulk_conductivity * np.clip(selected_dist / r_slope_inv + r_intercept, 0, 1)


100%|██████████████████████████████████████████████████████████████████████████████████| 282576/282576 [00:35<00:00, 7947.42it/s]


In [5]:
%%time

'''
    Construct the grid and add the conductances based on the closest atom (this is using the tree query method)
'''

#Grid spacing along dimensions
x_arr = np.arange(-1.0 * int(size_x / 2), 1.0 * int(size_x / 2), 1, dtype=int)
y_arr = np.arange(-1.0 * int(size_y / 2), 1.0 * int(size_y / 2), 1, dtype=int)
z_arr = np.arange(-1.0 * int(size_z / 2), 1.0 * int(size_z / 2), 1, dtype=int)

#Method for having nice tqdm progress bar
coords = np.array(list(itertools.product(x_arr,y_arr,z_arr)))

#Generate the KDTree
tree_pos = KDTree(system.positions)
tree_grid = KDTree(coords)

#Make the query, returns list of nearest neighbor for every point 
query = tree_grid.query_ball_tree(tree_pos, r=(r_max + 1.9)) #This does all querys between the two trees

#Reformat to a numpy via a pandas dataframe for ease of use since the list is inhomogenous originally
query_np = pd.DataFrame(query).to_numpy()

#Will store final conductivity map
conductivity_map = np.empty((len(query), 1))

#For any point that has nothing within maximum distance, we set to max bulk
conductivity_map[np.all(np.isnan(query_np), axis=1)] = bulk_conductivity

#Now we deal with the ones with a maximum distance
indices=np.arange(0,query_np.shape[0]) #Used to get the indices 

#Initialize a temp copy so that we can replace used up rows with NaNs 
temp_query = query_np.copy()

#We already did the all nan case so we ignore it, look at all rows with non-all nan
for index in reversed(range(query_np.shape[1])[1:]):

    #For easier use, we use the spliced query
    temp_query = temp_query[:,:index]

    #Construct the mask to look at rows with non NaN values
    mask = np.all(~np.isnan(temp_query), axis=1)
    
    #Compute the distances from given grid coords
    dist = np.linalg.norm(system.positions[temp_query[mask].astype(int)] - coords[mask][:,None,:], axis=2)
    real_dist = dist - radii[temp_query[mask].astype(int)] #Correct the distances with the atom's surface
    
    #Grab the smallest distances
    shortest_dist = np.min(real_dist, axis=1)
    
    #Update the respective conductivities
    conductivity_map[mask] = bulk_conductivity * np.clip(np.expand_dims(shortest_dist,-1) / r_slope_inv + r_intercept, 0, 1)

    #Change the used rows to all NaNs so we don't redo already done positions while ignoring certain neighbors
    temp_query[mask] = np.nan

test_1 = conductivity_map

CPU times: user 5.28 s, sys: 276 ms, total: 5.55 s
Wall time: 5.58 s


In [8]:
%%time
# #Grid spacing along dimensions
x_arr = np.arange(-1.0 * int(size_x / 2), 1.0 * int(size_x / 2), 1, dtype=int)
y_arr = np.arange(-1.0 * int(size_y / 2), 1.0 * int(size_y / 2), 1, dtype=int)
z_arr = np.arange(-1.0 * int(size_z / 2), 1.0 * int(size_z / 2), 1, dtype=int)

coords_x, coords_y, coords_z = np.meshgrid(x_arr, y_arr, z_arr, indexing='ij')

size_x = coords_x.shape[0]
size_y = coords_y.shape[1]
size_z = coords_z.shape[2]

coords = np.concatenate([np.expand_dims(coords_x,-1), np.expand_dims(coords_y, -1), np.expand_dims(coords_z, -1)], axis=3)
coords = coords.reshape(size_x * size_y * size_z, 3)

CPU times: user 2.66 ms, sys: 2.14 ms, total: 4.81 ms
Wall time: 3.92 ms


In [9]:
%%time

'''
    Construct the grid and add the conductances based on the closest atom (this is using the tree query method)
'''

#Generate the KDTree
tree_pos = KDTree(system.positions)
tree_grid = KDTree(coords)

#Make the query, returns list of nearest neighbor for every point 
query = tree_grid.query_ball_tree(tree_pos, r=(r_max + 1.9)) #This does all querys between the two trees

#Reformat to a numpy via a pandas dataframe for ease of use since the list is inhomogenous originally
query_np = pd.DataFrame(query).to_numpy()

#Useful variables for reshaping
entries = query_np.shape[0]
max_neighbors = query_np.shape[1]

#Reshape and create nan mask
query_np = query_np.reshape(entries * max_neighbors)
mask = ~np.isnan(query_np)
indices = query_np[mask].astype(int)

#Construct the proper grid coordinates 
grid_coords = np.arange(entries).repeat(max_neighbors)[mask]

#Compute non-nan distances (with radii corrections)
query_np[mask] = np.linalg.norm(system.positions[indices] - coords[grid_coords], axis=1) - radii[indices]

#Assign a large value for nan values
query_np[~mask] = 10

#Will store final conductivity map
conductivity_map = np.min(query_np.reshape(entries,max_neighbors), axis=1)
conductivity_map = bulk_conductivity * np.clip(conductivity_map / r_slope_inv + r_intercept, 0, 1)

interpolator = RegularGridInterpolator((x_arr, y_arr, z_arr), conductivity_map.reshape(size_x, size_y, size_z), bounds_error=False, fill_value=bulk_conductivity)

CPU times: user 2.51 s, sys: 334 ms, total: 2.84 s
Wall time: 2.85 s


In [13]:
'''
    Validate the conductivity map
    Directly checking the initial grid with the fast conductivity map is tricky due to reshaping
'''

i = 0
for pos in tqdm(coords):
    x,y,z = pos.astype(int)
    query = tree_pos.query_ball_point(pos, r=(r_max + 1.9))

    #Check that if this query is zero, then the conductivity must be 10.5
    if(len(query) == 0):
        assert conductivity_map[i] == 10.5
        # assert grid[x,y,z] == 10.5
    else:
        distance = np.linalg.norm(system.positions[query] - pos, axis=1) - radii[query]
        selected_distance = np.min(distance)
        selected_conductivity = bulk_conductivity * np.clip(selected_distance / r_slope_inv + r_intercept, 0, 1)
        
        assert conductivity_map[i] == selected_conductivity
        # assert grid[x,y,z] == selected_conductivity
        
    i += 1

100%|█████████████████████████████████████████████████████████████████████████████████| 282576/282576 [00:26<00:00, 10807.98it/s]
